# 02 — Train encoders on GPU (transductive, inductive, learning curve)

Each run is resumable (epoch checkpoints on Drive) and never retrained once finished. Families are trained one by one; ablations are opt-in. TRANSDUCTIVE encoders serve label-free diagnostics only; INDUCTIVE encoders (one per outer fold, held-out athletes removed from both domains) are the only ones used for NFL prediction.

In [ ]:
# install the latest main (torch untouched); restart the session first if nflbowl was already imported
%pip install -q "git+https://github.com/mybenkhadda/NFL_bowl.git@main"
%pip install -q --upgrade --force-reinstall --no-deps "git+https://github.com/mybenkhadda/NFL_bowl.git@main"
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

import nflbowl
from nflbowl import Experiment

print("nflbowl", nflbowl.__version__)                     # ≥ 0.2.1: fp32 contrastive losses under fp16 autocast
info = Experiment.device_info()
assert info.get("cuda_available"), "no GPU: Runtime → Change runtime type → GPU"

BUNDLE_DIR = Path("/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1")   # your bundle location
assert (BUNDLE_DIR / "bundle.json").is_file(), f"no bundle.json under {BUNDLE_DIR} (see 00_colab_quickstart Cell 4)"

# same name → reopens the experiment and its Drive checkpoints
exp = Experiment(data_dir=str(BUNDLE_DIR), output_dir="/content/drive/MyDrive/NFLBowl", config="colab_gpu", device="auto", name="f4_main")
exp.prepare()
exp.list_models()

In [ ]:
# primary encoder, then the other families (ablations=False by default)
training_table = exp.train("F4", resume=True)
display(training_table)
exp.train_all(families=True, ablations=False)

In [ ]:
# ablations + negative controls (F4_nophys, F4_noalign, F4_adv, F4_mmd, F4_category, F4_shuffled_phys) — optional
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    exp.train_all(families=False, ablations=True)

In [ ]:
# inductive encoders: F2 and F4 × outer folds (science.inductive_configs)
exp.train_all(regime="inductive")

In [ ]:
# learning curve against unlabeled NFL data (transductive F4 with 10 % and 30 % of the NFL windows)
for frac in exp.cfg.science.learning_curve_fractions:
    exp.train("F4", nfl_fraction=frac)

In [ ]:
# after a disconnect: continue every unfinished job
exp.resume()
exp.list_artifacts("training")